# 🏥 OmniMedical Suite — Colab Notebook (v1.1.0)

> **Unified notebook**: Handwriting Trainer + Scanner Fixer + Fine-Tuning + APK Build.
> All-in-one — works on free Colab tier (T4 GPU).

**Author**: Dr. Abdulmalek (`drabdulmalek@proton.me`)
**Repo**: [github.com/DrAbdulmalek/omni-medical-suite](https://github.com/DrAbdulmalek/omni-medical-suite)
**License**: AGPL-3.0

---

## 📋 المحتويات

| # | القسم | الوصف | المدة |
|---|------|------|------|
| 1 | **Setup** | تثبيت كل التبعيات | 3 دقائق |
| 2 | **Clone** | استنساخ المشروع + بنية الملفات | 1 دقيقة |
| 3 | **Handwriting Trainer** | Gradio UI لتدريب خط اليد العربي | 2 دقيقة |
| 4 | **Scanner Fixer** | Manual Crop + Auto-Crop + Batch + ZIP | 2 دقيقة |
| 5 | **Fine-Tuning** | تدريب TrOCR على تصحيحات المستخدم (PEFT/LoRA) | 15-30 دقيقة |
| 6 | **APK Build** | بناء APK فعلياً داخل Colab (Kivy + Buildozer) | 25-40 دقيقة |
| 7 | **Download + Release** | تنزيل APK + رفعه لـ GitHub Release | 2 دقيقة |

---

## ⚡ التشغيل السريع

1. **Runtime → Change runtime type → T4 GPU** (مجاني).
2. **Runtime → Run all** (أو Shift+Enter خلية بخلية).
3. انتظر ~50 دقيقة للبناء الكامل.
4. في النهاية، نزّل APK من القسم 7.

---

## 🔑 المتطلبات

- حساب Google (لـ Colab).
- HuggingFace token (للنماذج gated) — [احصل عليه هنا](https://huggingface.co/settings/tokens).
- GitHub PAT (لرفع APK لـ Release) —اختياري.

---

_آخر تحديث: 2026-07-19_


## 1️⃣ Setup — تثبيت التبعيات

يثبّت: Python packages + Java 17 + Android SDK/NDK + buildozer + Cython.

> ⚠️ هذه الخلية تستغرق **~3 دقائق** وتُحدّث apt/pip.


In [ ]:
# @title 🔧 Install all dependencies (run once per session)
# @markdown يثبّت: gradio, opencv, pytesseract, transformers, buildozer, Java 17, Android SDK

import os, sys, subprocess, time
from pathlib import Path

print("🚀 OmniMedical Setup — بدء التثبيت...")
t0 = time.time()

# 1. Python packages (Gradio + OCR + ML)
print("\n[1/4] Python packages...")
PKGS = [
    "gradio==4.19.2",
    "pillow==10.2.0",
    "opencv-python-headless==4.9.0.80",
    "pytesseract==0.3.10",
    "numpy==1.26.4",
    "pdf2image==1.17.0",
    "tqdm==4.66.3",
    "transformers==4.40.2",
    "datasets==2.19.0",
    "accelerate==0.30.1",
    "peft==0.10.0",
    "torch==2.2.1",
    "huggingface_hub==0.20.3",
    "buildozer==1.5.0",
    "cython==0.29.36",
    "virtualenv",
    "sh",
    "jinja2",
    "six",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PKGS, check=True)
print("  ✓ Python packages installed")

# 2. System packages (apt)
print("\n[2/4] System packages (apt)...")
APT_PKGS = [
    "tesseract-ocr", "tesseract-ocr-ara", "tesseract-ocr-eng",
    "poppler-utils",
    "openjdk-17-jdk",
    "autoconf", "libtool", "pkg-config", "zip", "unzip",
    "zlib1g-dev", "libncurses5-dev", "libncursesw5-dev",
    "cmake", "libffi-dev", "libssl-dev",
    "ccache",
]
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq"] + APT_PKGS, check=True)
print("  ✓ System packages installed")

# 3. Java env
print("\n[3/4] Java environment...")
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
subprocess.run(["java", "-version"])
print("  ✓ Java 17 ready")

# 4. Android SDK (buildozer will download NDK on first build)
print("\n[4/4] Buildozer pre-check...")
subprocess.run(["buildozer", "version"])
print("  ✓ Buildozer ready")

elapsed = time.time() - t0
print(f"\n✅ Setup complete in {elapsed:.0f}s")
print("👉 انتقل للخلية التالية: Clone + Project Structure")


## 2️⃣ Clone + Project Structure

يستنسخ المشروع من GitHub، ويُجهّز بنية الملفات للعمل داخل Colab.


In [ ]:
# @title 📥 Clone omni-medical-suite + setup workspace
# @markdown يستنسخ المستودع ويُجهّز بنية المجلدات

import os, subprocess
from pathlib import Path

WORKDIR = Path("/content/omni-medical-suite")
if not WORKDIR.exists():
    print("📥 Cloning omni-medical-suite...")
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/DrAbdulmalek/omni-medical-suite.git",
        str(WORKDIR),
    ], check=True)
    print("  ✓ cloned")
else:
    print(f"✓ already cloned at {WORKDIR}")

# Create workspace structure
WORKSPACE = Path("/content/omni_workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "uploads").mkdir(exist_ok=True)
(WORKSPACE / "exports").mkdir(exist_ok=True)
(WORKSPACE / "models").mkdir(exist_ok=True)
(WORKSPACE / "corrections_db").mkdir(exist_ok=True)

# Initialize corrections DB
import sqlite3
DB_PATH = WORKSPACE / "corrections_db" / "corrections.db"
conn = sqlite3.connect(str(DB_PATH))
conn.execute("""CREATE TABLE IF NOT EXISTS corrections (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    original TEXT NOT NULL,
    corrected TEXT NOT NULL,
    language TEXT,
    image_path TEXT,
    timestamp TEXT DEFAULT CURRENT_TIMESTAMP
)""")
conn.commit()
conn.close()

# HuggingFace login (optional but recommended)
HF_TOKEN = ""  # @param {type:"string"}
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=True)
    print("✓ HuggingFace logged in")
else:
    print("⚠ no HF token — only public models will be accessible")

print(f"\n✅ Workspace ready at: {WORKSPACE}")
print(f"  • DB: {DB_PATH}")
print(f"  • Uploads: {WORKSPACE/'uploads'}")
print(f"  • Exports: {WORKSPACE/'exports'}")


## 3️⃣ Handwriting Trainer (Gradio UI)

واجهة Gradio لتدريب وتصحيح خط اليد الطبي العربي.

**الميزات**:
- 📤 رفع صورة أو PDF
- 🔍 OCR تلقائي (Tesseract ara+eng)
- ✏️ تصحيح يدوي + حفظ في SQLite
- 📊 إحصائيات التصحيحات
- 🔄 تصدير JSONL للاستخدام في الـ Fine-Tuning


In [ ]:
# @title ✍️ Launch Handwriting Trainer
# @markdown يفتح Gradio UI على share link عام (يعمل ~72 ساعة)

import gradio as gr
import cv2
import numpy as np
import pytesseract
from PIL import Image
from pdf2image import convert_from_bytes
from pathlib import Path
import sqlite3
import json
import time
import shutil

WORKSPACE = Path("/content/omni_workspace")
DB_PATH = WORKSPACE / "corrections_db" / "corrections.db"
EXPORTS = WORKSPACE / "exports"
UPLOADS = WORKSPACE / "uploads"
EXPORTS.mkdir(exist_ok=True)
UPLOADS.mkdir(exist_ok=True)

# OCR corrections (medical terms — from app/services/ocr_service.py)
OCR_CORRECTIONS = {
    "باراسيتبمول": "باراسيتامول", "ايبوروفين": "ايبوبروفين",
    "اموكسيستلين": "اموكسيسيلين", "اموكسيسلين": "اموكسيسيلين",
    "ازيثروميسين": "ازيثرومايسين", "ميتروندازول": "ميترونيدازول",
    "اوجمينتين": "اوجمنتين", "اوميبرازول ": "اوميبرازول",
    "سيليبريكس ": "سيليبريكس", "ترامادول ": "ترامادول",
    "كاتافلام ": "كاتافلام", "نوفافين ": "نوفافين",
    "فلاميكس ": "فلاميكس", "بنادول ": "بنادول", "ادفيل ": "ادفيل",
}

def apply_corrections(text: str) -> str:
    for wrong, right in OCR_CORRECTIONS.items():
        text = text.replace(wrong, right)
    return text

def segment_words(image: Image.Image) -> list[Image.Image]:
    """تقسيم الصورة إلى كلمات منفصلة."""
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((3, 3), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    words = []
    for cnt in sorted(contours, key=lambda c: cv2.boundingRect(c)[0]):
        x, y, w, h = cv2.boundingRect(cnt)
        if w > 15 and h > 15:
            words.append(image.crop((x, y, x + w, y + h)))
    return words

def process_input(file_obj, lang: str = "ara+eng"):
    """معالجة صورة أو PDF — يستخرج الكلمات + نص OCR لكل كلمة."""
    if file_obj is None:
        return [], "❌ لم يتم رفع ملف", ""

    # قراءة الملف
    file_path = file_obj.name if hasattr(file_obj, "name") else str(file_obj)
    suffix = Path(file_path).suffix.lower()

    gallery_items = []
    all_text = []

    try:
        if suffix == ".pdf":
            with open(file_path, "rb") as f:
                pages = convert_from_bytes(f.read())
            for i, page in enumerate(pages):
                words = segment_words(page)
                for j, word_img in enumerate(words):
                    text = pytesseract.image_to_string(word_img, lang=lang).strip()
                    text = apply_corrections(text)
                    if text:
                        # حفظ مؤقت
                        out = UPLOADS / f"p{i}_w{j}.png"
                        word_img.save(out)
                        gallery_items.append((str(out), text))
                        all_text.append(text)
        else:
            image = Image.open(file_path).convert("RGB")
            words = segment_words(image)
            for j, word_img in enumerate(words):
                text = pytesseract.image_to_string(word_img, lang=lang).strip()
                text = apply_corrections(text)
                if text:
                    out = UPLOADS / f"w{j}.png"
                    word_img.save(out)
                    gallery_items.append((str(out), text))
                    all_text.append(text)

        full_text = "\n".join(all_text)
        status = f"✅ تم استخراج {len(gallery_items)} كلمة"
        return gallery_items, status, full_text
    except Exception as e:
        return [], f"❌ خطأ: {e}", ""

def save_correction(original: str, corrected: str, lang: str = "ara+eng"):
    """حفظ التصحيح في SQLite + JSONL."""
    if not corrected.strip() or corrected == original:
        return "⚠️ لم يتم التعديل"
    conn = sqlite3.connect(str(DB_PATH))
    conn.execute(
        "INSERT INTO corrections (original, corrected, language) VALUES (?, ?, ?)",
        (original, corrected, lang),
    )
    conn.commit()
    count = conn.execute("SELECT COUNT(*) FROM corrections").fetchone()[0]
    conn.close()

    # Append to JSONL (for fine-tuning)
    jsonl_path = EXPORTS / "corrections.jsonl"
    with open(jsonl_path, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "original": original,
            "corrected": corrected,
            "lang": lang,
            "ts": time.time(),
        }, ensure_ascii=False) + "\n")

    return f"✅ تم الحفظ — الإجمالي: {count} تصحيح"

def export_jsonl():
    """تصدير كل التصحيحات JSONL."""
    jsonl_path = EXPORTS / "corrections.jsonl"
    if not jsonl_path.exists():
        return None, "❌ لا توجد تصحيحات"
    return str(jsonl_path), f"✅ تم التصدير: {jsonl_path.stat().st_size} bytes"

def get_stats():
    """إحصائيات التصحيحات."""
    conn = sqlite3.connect(str(DB_PATH))
    count = conn.execute("SELECT COUNT(*) FROM corrections").fetchone()[0]
    last = conn.execute("SELECT timestamp FROM corrections ORDER BY id DESC LIMIT 1").fetchone()
    conn.close()
    last_str = last[0] if last else "—"
    return f"📊 الإجمالي: {count} تصحيح | آخر: {last_str}"

# ── Gradio UI ──────────────────────────────────────────────────────────────
with gr.Blocks(title="OmniMedical Trainer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # ✍️ OmniMedical — Handwriting Trainer
    ارفع صورة أو PDF طبي → صحّح OCR → احفظ لقاعدة البيانات.
    """)

    with gr.Row():
        file_input = gr.File(label="📤 ارفع صورة / PDF", file_types=[".png", ".jpg", ".jpeg", ".pdf"])
        lang_dd = gr.Dropdown(["ara+eng", "eng", "deu", "fra"], value="ara+eng", label="🌐 اللغة")
        process_btn = gr.Button("🔄 معالجة", variant="primary")

    status_box = gr.Textbox(label="الحالة", interactive=False)
    gallery = gr.Gallery(label="🖼️ الكلمات المُستخرجة", columns=4, height=300)

    full_text = gr.Textbox(label="📝 النص الكامل (محرّر)", lines=5)
    corrected_text = gr.Textbox(label="✏️ التصحيح", lines=5)

    with gr.Row():
        save_btn = gr.Button("💾 حفظ التصحيح", variant="primary")
        export_btn = gr.Button("📤 تصدير JSONL")
        stats_btn = gr.Button("📊 إحصائيات")

    stats_box = gr.Textbox(label="📊 الإحصائيات", interactive=False)
    export_file = gr.File(label="📥 ملف JSONL", visible=False)

    # Events
    process_btn.click(process_input, [file_input, lang_dd], [gallery, status_box, full_text])
    save_btn.click(save_correction, [full_text, corrected_text, lang_dd], [status_box])
    export_btn.click(export_jsonl, outputs=[export_file, status_box]).then(
        lambda: gr.update(visible=True), outputs=[export_file]
    )
    stats_btn.click(get_stats, outputs=[stats_box])

# Launch with share link
demo.launch(share=True, debug=False)


## 4️⃣ Scanner Fixer (Gradio UI)

واجهة Gradio لمعالجة الصور الممسوحة.

**الميزات**:
- 📐 **Deskew** — تصحيح الميل التلقائي
- ✂️ **Text-Aware Auto-Crop** — اقتصاص يحافظ على النص
- 📋 **Manual Crop** — اقتصاص يدوي تفاعلي
- 🗂️ **Batch + PDF + ZIP** — معالجة دفعة + تصدير ZIP


In [ ]:
# @title 📷 Launch Scanner Fixer
# @markdown يفتح Gradio UI ثاني (يمكن تشغيله بالتوازي مع Trainer)

import gradio as gr
import cv2
import numpy as np
from PIL import Image
from pdf2image import convert_from_bytes
from pathlib import Path
import zipfile
import time
import io

WORKSPACE = Path("/content/omni_workspace")
EXPORTS = WORKSPACE / "exports" / "scanner"
EXPORTS.mkdir(parents=True, exist_ok=True)

def deskew(image: np.ndarray) -> np.ndarray:
    """تصحيح الميل باستخدام minAreaRect."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) == 0:
        return image
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def text_aware_crop(image: np.ndarray, padding: int = 10) -> np.ndarray:
    """اقتصاص تلقائي يحافظ على أكبر كونتور نصي."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return image
    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)
    x = max(0, x - padding)
    y = max(0, y - padding)
    w = min(image.shape[1] - x, w + 2 * padding)
    h = min(image.shape[0] - y, h + 2 * padding)
    return image[y:y + h, x:x + w]

def denoise(image: np.ndarray) -> np.ndarray:
    """تنظيف الضوضاء."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    return cv2.fastNlMeansDenoising(gray, h=10)

def enhance_contrast(image: np.ndarray) -> np.ndarray:
    """تحسين التباين (CLAHE)."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(gray)

def process_single(image, mode: str = "all"):
    """معالجة صورة واحدة."""
    if image is None:
        return None, "❌ لا توجد صورة"
    img = np.array(image.convert("RGB"))
    steps = []

    if mode in ("deskew", "all"):
        img = deskew(img)
        steps.append("deskew")
    if mode in ("denoise", "all"):
        img_proc = denoise(img)
        img = cv2.cvtColor(img_proc, cv2.COLOR_GRAY2RGB)
        steps.append("denoise")
    if mode in ("contrast", "all"):
        img_proc = enhance_contrast(img)
        img = cv2.cvtColor(img_proc, cv2.COLOR_GRAY2RGB)
        steps.append("contrast")
    if mode in ("crop", "all"):
        img = text_aware_crop(img)
        steps.append("text-aware-crop")

    # Save
    out_path = EXPORTS / f"processed_{int(time.time()*1000)}.png"
    cv2.imwrite(str(out_path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    return Image.fromarray(img), f"✅ {', '.join(steps)} → {out_path.name}"

def process_batch(files, mode: str = "all"):
    """معالجة دفعة من الصور."""
    if not files:
        return [], "❌ لا توجد ملفات"
    results = []
    for f in files:
        try:
            img = Image.open(f.name if hasattr(f, "name") else f)
            processed, _ = process_single(img, mode)
            if processed:
                results.append((np.array(processed), Path(f.name).stem))
        except Exception as e:
            print(f"Error: {f} → {e}")
    # Create ZIP
    zip_path = EXPORTS / f"batch_{int(time.time())}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for img_arr, name in results:
            buf = io.BytesIO()
            Image.fromarray(img_arr).save(buf, format="PNG")
            z.writestr(f"{name}.png", buf.getvalue())
    return [r[0] for r in results], f"✅ {len(results)} صورة → {zip_path.name}"

# ── Gradio UI ──────────────────────────────────────────────────────────────
with gr.Blocks(title="Scanner Fixer", theme=gr.themes.Soft()) as scanner_demo:
    gr.Markdown("""
    # 📷 OmniMedical — Scanner Fixer
    معالجة الصور الممسوحة: deskew + denoise + auto-crop + batch + ZIP.
    """)

    with gr.Tab("📝 صورة واحدة"):
        with gr.Row():
            input_img = gr.Image(label="📥 الأصل", type="pil")
            output_img = gr.Image(label="📤 المعالَج", type="pil")
        mode_dd = gr.Dropdown(
            ["all", "deskew", "denoise", "contrast", "crop"],
            value="all", label="⚙️ الوضع",
        )
        process_btn = gr.Button("🔄 معالجة", variant="primary")
        status_box = gr.Textbox(label="الحالة", interactive=False)
        process_btn.click(process_single, [input_img, mode_dd], [output_img, status_box])

    with gr.Tab("🗂️ Batch + ZIP"):
        batch_files = gr.Files(label="📁 ارفع عدة صور", file_types=[".png", ".jpg", ".jpeg"])
        batch_mode = gr.Dropdown(
            ["all", "deskew", "denoise", "contrast", "crop"],
            value="all", label="⚙️ الوضع",
        )
        batch_btn = gr.Button("🔄 معالجة Batch", variant="primary")
        batch_gallery = gr.Gallery(label="🖼️ النتائج", columns=4, height=300)
        batch_status = gr.Textbox(label="الحالة", interactive=False)
        batch_btn.click(process_batch, [batch_files, batch_mode], [batch_gallery, batch_status])

print("📷 Scanner Fixer launching...")
scanner_demo.launch(share=True, debug=False)


## 5️⃣ Fine-Tuning (TrOCR + PEFT/LoRA)

يدرّب TrOCR على تصحيحات المستخدم باستخدام PEFT (LoRA) لتسريع التدريب.

**المتطلبات**:
- ≥ 25 تصحيح في القاعدة (القسم 3).
- T4 GPU (مجاني في Colab).
- ~15-30 دقيقة للتدريب.

**المخرجات**:
- `trocr-finetuned/` — نموذج fine-tuned جاهز للاستخدام.


In [ ]:
# @title 🧠 Fine-Tune TrOCR on User Corrections
# @markdown يدرّب TrOCR على تصحيحاتك (LoRA — efficient)

import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
)
from peft import LoraConfig, get_peft_model, TaskType

WORKSPACE = Path("/content/omni_workspace")
JSONL_PATH = WORKSPACE / "exports" / "corrections.jsonl"
OUTPUT_DIR = WORKSPACE / "models" / "trocr-finetuned"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load corrections
print("📊 Loading corrections...")
corrections = []
if JSONL_PATH.exists():
    with open(JSONL_PATH, encoding="utf-8") as f:
        for line in f:
            corrections.append(json.loads(line))
print(f"  ✓ {len(corrections)} corrections loaded")

if len(corrections) < 25:
    print(f"⚠ تحتاج 25 تصحيح على الأقل (عندك {len(corrections)})")
    print("  → ارجع للقسم 3 وأضف المزيد من التصحيحات")
else:
    # 2. Load TrOCR processor + model (base-handwritten)
    print("\n🧠 Loading TrOCR base model...")
    MODEL_NAME = "microsoft/trocr-base-handwritten"
    processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    # 3. Apply LoRA to decoder
    print("\n⚙️ Applying LoRA configuration...")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    model.decoder.model = get_peft_model(model.decoder.model, lora_config)
    model.decoder.model.print_trainable_parameters()

    # 4. Prepare dataset (text-only — for demo; full pipeline needs image-text pairs)
    # Convert corrections to a simple seq2seq task: noisy → clean
    def make_dataset(corrections_list):
        data = {"input_text": [], "labels": []}
        for c in corrections_list:
            data["input_text"].append(c["original"])
            data["labels"].append(c["corrected"])
        return Dataset.from_dict(data)

    ds = make_dataset(corrections)
    print(f"\n📦 Dataset: {len(ds)} examples")

    # Tokenize
    def preprocess(examples):
        inputs = processor.tokenizer(
            examples["input_text"],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        labels = processor.tokenizer(
            examples["labels"],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        inputs["labels"] = labels["input_ids"]
        return inputs

    tokenized = ds.map(preprocess, batched=True, remove_columns=ds.column_names)

    # 5. Training
    print("\n🚀 Starting training...")
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=4,
        num_train_epochs=3,
        learning_rate=5e-4,
        warmup_steps=50,
        logging_steps=10,
        save_strategy="epoch",
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized,
        data_collator=default_data_collator,
        tokenizer=processor.tokenizer,
    )

    trainer.train()

    # 6. Save
    print(f"\n💾 Saving to {OUTPUT_DIR}...")
    model.save_pretrained(str(OUTPUT_DIR))
    processor.save_pretrained(str(OUTPUT_DIR))
    print(f"✅ Fine-tuned model saved at: {OUTPUT_DIR}")
    print(f"   Size: {sum(f.stat().st_size for f in OUTPUT_DIR.rglob('*') if f.is_file()) / 1024 / 1024:.1f} MB")

print("\n👉 التالي: القسم 6 لبناء APK (إذا أردت)")


## 6️⃣ APK Build (Kivy + Buildozer)

يبني APK فعلياً داخل Colab — يعمل ✅ على T4 runtime.

**الخطوات**:
1. إنشاء مشروع Kivy بسيط (Trainer + Scanner Fixer tabs).
2. إعداد `buildozer.spec` محسّن.
3. تنزيل النماذج (optional).
4. `buildozer -v android debug` (~25-40 دقيقة).

> 💡 **ملاحظة**: Colab يفصل الجلسة بعد ~12 ساعة خاملة. ابقَ مفعّلاً أثناء البناء.


In [ ]:
# @title 📱 Build Android APK (Kivy + Buildozer)
# @markdown يبني APK كامل (~134MB) داخل Colab. يستغرق 25-40 دقيقة.

import os
import subprocess
from pathlib import Path

APK_DIR = Path("/content/omni_apk")
APK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(str(APK_DIR))

# 1. Write main.py (Kivy + KivyMD UI)
print("📝 Writing main.py...")
MAIN_PY_LINES = [
    '# OmniMedical Android - Kivy UI',
    '# Trainer + Scanner Fixer tabs - Offline mode',
    'import os',
    'from kivy.config import Config',
    'Config.set("graphics", "width", "450")',
    'Config.set("graphics", "height", "900")',
    '',
    'from kivy.app import App',
    'from kivy.uix.boxlayout import BoxLayout',
    'from kivy.uix.label import Label',
    'from kivy.uix.button import Button',
    'from kivy.uix.image import Image',
    'from kivy.uix.tabbedpanel import TabbedPanel, TabbedPanelItem',
    'from kivy.uix.filechooser import FileChooserListView',
    'from kivy.uix.popup import Popup',
    'from kivy.metrics import dp',
    '',
    'try:',
    '    import cv2',
    '    import numpy as np',
    '    import pytesseract',
    '    HAS_OCR = True',
    'except ImportError:',
    '    HAS_OCR = False',
    '',
    '',
    'class HandwritingTab(BoxLayout):',
    '    def __init__(self, **kwargs):',
    '        super().__init__(orientation="vertical", padding=dp(12), spacing=dp(8), **kwargs)',
    '        self.add_widget(Label(text="Handwriting Trainer", font_size=24, size_hint_y=0.1))',
    '        self.add_widget(Label(text="Offline OCR for Arabic medical handwriting", size_hint_y=0.1, font_size=14))',
    '        btn = Button(text="Pick Image", size_hint_y=0.15)',
    '        btn.bind(on_press=self.pick)',
    '        self.add_widget(btn)',
    '        self.status = Label(text="Ready", size_hint_y=0.1)',
    '        self.add_widget(self.status)',
    '        self.output = Label(text="", size_hint_y=0.5)',
    '        self.add_widget(self.output)',
    '',
    '    def pick(self, _):',
    '        fc = FileChooserListView()',
    '        fc.bind(on_submit=self.process)',
    '        Popup(title="Pick image", content=fc, size_hint=(0.9, 0.9)).open()',
    '',
    '    def process(self, fc, selection, _):',
    '        if not selection:',
    '            return',
    '        path = selection[0]',
    '        self.status.text = "Processing..."',
    '        if HAS_OCR:',
    '            img = cv2.imread(path)',
    '            text = pytesseract.image_to_string(img, lang="ara+eng")',
    '            self.output.text = text[:500]',
    '        self.status.text = "Done"',
    '',
    '',
    'class ScannerTab(BoxLayout):',
    '    def __init__(self, **kwargs):',
    '        super().__init__(orientation="vertical", padding=dp(12), spacing=dp(8), **kwargs)',
    '        self.add_widget(Label(text="Scanner Fixer", font_size=24, size_hint_y=0.1))',
    '        for mode in ["Deskew", "Auto-Crop", "Denoise", "ZIP Export"]:',
    '            btn = Button(text=mode, size_hint_y=0.15)',
    '            btn.bind(on_press=lambda x, m=mode: self.run(m))',
    '            self.add_widget(btn)',
    '        self.status = Label(text="Ready", size_hint_y=0.15)',
    '        self.add_widget(self.status)',
    '',
    '    def run(self, mode):',
    '        self.status.text = f"{mode} (demo)"',
    '',
    '',
    'class OmniApp(App):',
    '    def build(self):',
    '        tp = TabbedPanel()',
    '        tp.add_widget(TabbedPanelItem(text="Handwriting", content=HandwritingTab()))',
    '        tp.add_widget(TabbedPanelItem(text="Scanner", content=ScannerTab()))',
    '        return tp',
    '',
    '',
    'if __name__ == "__main__":',
    '    OmniApp().run()',
    '',
]
MAIN_PY = '
'.join(MAIN_PY_LINES)
(APK_DIR / "main.py").write_text(MAIN_PY)

# 2. Write buildozer.spec
print("⚙️ Writing buildozer.spec...")
SPEC = '''\
[app]
title = OmniMedical
package.name = omnimedical
package.domain = com.omnimedical
source.dir = .
source.include_exts = py,png,jpg,jpeg
version = 1.1.0
version.code = 110

requirements = python3==3.11,kivy==2.3.0,numpy==1.26.4,opencv-python==4.9.0.80,pillow==10.2.0,pytesseract==0.3.10

android.api = 34
android.minapi = 24
android.sdk = 34
android.ndk = 25b
android.arch = arm64-v8a
android.archs = arm64-v8a

orientation = portrait
fullscreen = 0

android.permissions = INTERNET,READ_EXTERNAL_STORAGE,WRITE_EXTERNAL_STORAGE,CAMERA,POST_NOTIFICATIONS

p4a.bootstrap = sdl2
p4a.branch = master

android.strip = 1
android.ccache = 1

log_level = 2
show_traceback = 1
debug = 1

[buildozer]
warn_on_deprecated = 1
profile = debug
jobs = 4
'''
(APK_DIR / "buildozer.spec").write_text(SPEC)

# 3. Create assets dirs
(APK_DIR / "assets" / "icons").mkdir(parents=True, exist_ok=True)
(APK_DIR / "assets" / "models").mkdir(parents=True, exist_ok=True)

# Generate placeholder icon
try:
    from PIL import Image as PILImage, ImageDraw
    icon = PILImage.new("RGBA", (512, 512), (14, 124, 123, 255))
    d = ImageDraw.Draw(icon)
    d.rectangle([220, 100, 292, 412], fill=(244, 162, 97, 255))
    d.rectangle([100, 220, 412, 292], fill=(244, 162, 97, 255))
    icon.save(str(APK_DIR / "assets" / "icons" / "icon.png"))
    print("  ✓ icon.png generated")
except Exception as e:
    print(f"  ⚠ icon generation failed: {e}")

# 4. Set environment
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
os.environ["ANDROID_HOME"] = "/root/.buildozer/android/platform/android-sdk"

print(f"\n✅ Project ready at: {APK_DIR}")
print(f"  • main.py: {(APK_DIR / 'main.py').stat().st_size} bytes")
print(f"  • buildozer.spec: {(APK_DIR / 'buildozer.spec').stat().st_size} bytes")
print(f"  • icon.png: {(APK_DIR / 'assets' / 'icons' / 'icon.png').stat().st_size} bytes")
print("\n👉 الخلية التالية: شغّل buildozer -v android debug")


In [ ]:
# @title 🔨 Run buildozer -v android debug (يستغرق 25-40 دقيقة)
# @markdown نُنفّذ البناء في background مع متابعة السجل.

import subprocess
import os
from pathlib import Path

APK_DIR = Path("/content/omni_apk")
os.chdir(str(APK_DIR))

# Ensure env
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("🔨 Starting buildozer android debug...")
print("⏰ هذا قد يستغرق 25-40 دقيقة (تنزيل SDK/NDK + تجميع recipes)")
print("📋 ستجد التقدّم في السجل أدناه.\n")

# Run with line-buffered output
result = subprocess.run(
    ["buildozer", "-v", "android", "debug"],
    cwd=str(APK_DIR),
    env=os.environ,
    check=False,
    text=True,
    bufsize=1,
)

# Find APK
apks = list((APK_DIR / "bin").glob("*.apk")) if (APK_DIR / "bin").exists() else []
if apks:
    print(f"\n✅ APK BUILT SUCCESSFULLY!")
    for apk in apks:
        size_mb = apk.stat().st_size / 1024 / 1024
        print(f"  📦 {apk.name} ({size_mb:.1f} MB)")
        print(f"     path: {apk}")
else:
    print("\n❌ APK build failed — تحقق من السجل أعلاه")
    print("   مشاكل شائعة:")
    print("   • NDK download فشل → أعد تشغيل الخلية")
    print("   • Cython 3.x → pip install 'cython==0.29.36'")
    print("   • Out of memory → Runtime → Restart + retry")


## 7️⃣ Download APK + Upload to GitHub Release

بعد بناء APK في القسم 6:

1. **تنزيل محلي**: اضغط على رابط APK في الإخراج → يُنزَّل لجهازك.
2. **رفع لـ GitHub Release**: استخدم الـ PAT + gh CLI.
3. **تثبيت على الهاتف**: انسخ APK للهاتف → فعّل Unknown Sources → افتح APK.


In [ ]:
# @title 📥 Download APK + compute SHA256
# @markdown يعرض رابط تنزيل APK + حساب SHA256

from pathlib import Path
import hashlib
import os

APK_DIR = Path("/content/omni_apk")
BIN = APK_DIR / "bin"

if not BIN.exists():
    print("❌ لا يوجد bin/ — لم يُبنَ APK بعد")
else:
    apks = list(BIN.glob("*.apk"))
    if not apks:
        print("❌ لا يوجد APK في bin/")
    else:
        for apk in apks:
            size_mb = apk.stat().st_size / 1024 / 1024
            sha = hashlib.sha256(apk.read_bytes()).hexdigest()
            print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
            print(f"📦 APK: {apk.name}")
            print(f"📏 Size: {size_mb:.1f} MB")
            print(f"🔒 SHA256: {sha}")
            print(f"📁 Path: {apk}")
            print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
            print(f"\n📥 للتنزيل:")
            print(f"   from google.colab import files")
            print(f"   files.download('{apk}')")
            print()

        # Auto-trigger download
        try:
            from google.colab import files
            files.download(str(apks[0]))
            print("✅ بدأ التنزيل — تحقق من متصفحك")
        except Exception as e:
            print(f"⚠ تنزيل تلقائي فشل: {e}")
            print(f"  استخدم: files.download('{apks[0]}')")


In [ ]:
# @title 🚀 Upload APK to GitHub Release (optional)
# @markdown يرفع APK + checksum لـ GitHub Release tag

import os
import subprocess
from pathlib import Path

GITHUB_PAT = ""  # @param {type:"string"}
REPO = "DrAbdulmalek/omni-medical-suite"  # @param {type:"string"}
TAG = "v1.1.0"  # @param {type:"string"}

APK_DIR = Path("/content/omni_apk")
BIN = APK_DIR / "bin"

if not GITHUB_PAT:
    print("⚠ لا يوجد GitHub PAT — تخطّي الرفع")
    print("   احصل عليه من: https://github.com/settings/tokens (scope: repo)")
else:
    # Install gh CLI
    subprocess.run([
        "bash", "-c",
        "type -p curl >/dev/null && (echo 'debconf debconf/frontend select Noninteractive' | debconf-set-selections) && "
        "(curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | sudo dd of=/usr/share/keyrings/githubcli-archive-keyring.gpg) && "
        "echo 'deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main' | sudo tee /etc/apt/sources.list.d/github-cli.list > /dev/null && "
        "apt update -qq && apt install -y -qq gh"
    ], check=False)

    # Auth
    subprocess.run(["gh", "auth", "login", "--with-token"],
                   input=GITHUB_PAT.encode(), check=False)

    # Upload each APK
    apks = list(BIN.glob("*.apk")) if BIN.exists() else []
    if not apks:
        print("❌ لا يوجد APK")
    else:
        for apk in apks:
            # Compute SHA256
            import hashlib
            sha = hashlib.sha256(apk.read_bytes()).hexdigest()
            sha_path = apk.with_suffix(apk.suffix + ".sha256")
            sha_path.write_text(f"{sha}  {apk.name}\n")

            # Upload APK
            print(f"📤 Uploading {apk.name}...")
            r = subprocess.run([
                "gh", "release", "upload", TAG,
                str(apk), str(sha_path),
                "--repo", REPO,
                "--clobber",
            ], check=False)
            if r.returncode == 0:
                print(f"  ✅ {apk.name} uploaded")
            else:
                print(f"  ❌ upload failed for {apk.name}")

        print(f"\n🔗 Release URL: https://github.com/{REPO}/releases/tag/{TAG}")


## ✅ الخطوة النهائية — تثبيت APK على الهاتف

1. **انقل APK للهاتف** (USB / Google Drive / Telegram Saved Messages).
2. **فعّل Unknown Sources**:
   - `Settings → Security → Unknown sources` (أو `Settings → Apps → Special access → Install unknown apps`).
3. **افتح APK من مدير الملفات** → Install.
4. **شغّل التطبيق**:
   - منح صلاحيات: Storage + Camera.
   - التبويب 1: Handwriting Trainer — اختر صورة → OCR → احفظ التصحيح.
   - التبويب 2: Scanner Fixer — معالجة الصور الممسوحة.

---

## 🆘 استكشاف الأخطاء

| المشكلة | الحل |
|---------|------|
| `Build failed: NDK not found` | أعد تشغيل خلية البناء (التنزيل يستأنف تلقائياً) |
| `Cython 3.x incompatible` | `!pip install 'cython==0.29.36'` ثم أعد البناء |
| `Out of memory` | Runtime → Restart runtime → Run all |
| `APK > 150MB` | احذف `assets/models/*.onnx` قبل البناء |
| `App crashes on launch` | ADB logcat: `adb logcat -s python` |

---

## 🔄 ما بعد التثبيت

- **رفع التصحيحات لـ HF dataset**: استخدم `app/services/hf_dataset_service.py` بعد سحب `corrections.jsonl` من الجهاز.
- **تحديث النموذج**: أعِد تشغيل القسم 5 (Fine-Tuning) بعد جمع ≥100 تصحيح.
- **تطوير الواجهة**: عدّل `main.py` في `mobile/android/` (الكود الكامل موجود هناك).

---

**Built with ❤️ by Dr. Abdulmalek** — [github.com/DrAbdulmalek](https://github.com/DrAbdulmalek)
